In [1]:
# imports

# to get my token from .env
from dotenv import load_dotenv

# get token
import os

# handle requests 
import requests 

# db adapter
import psycopg2

# insert data into staging tables
from psycopg2.extras import execute_values, Json

In [2]:
# get token from .env

# load .env
load_dotenv()

# get token
TOKEN = os.getenv("GITHUB_TOKEN")

# get DB name
DB_NAME = os.getenv("DB_NAME")

# get DB password
PASSWORD = os.getenv("PASSWORD")

# ensure env vars exists
print("TOKEN exists:", TOKEN is not None)
print("DB_NAME exists:", DB_NAME is not None)
print("PASSWORD exists:", PASSWORD is not None)

TOKEN exists: True
DB_NAME exists: True
PASSWORD exists: True


In [3]:
# declare headers with TOKEN

headers = {
    "Authorization": f"Bearer {TOKEN}"
}

# rename GitHub REST API columns to match table columns
rename_users_map = {
    "id": "user_id",
    "type": "user_type",
    "name": "user_name",
    "location": "user_location",
    "following": "user_following"
}

rename_repos_map = {
    "id": "repo_id",
    "language": "repo_language",
    "size": "repo_size",
    "name": "repo_name"
}

rename_commits_map = {
    "message": "repo_message"
}

rename_issues_map = {
    "id": "issue_id",
    "comments": "issue_comments",
    "number": "issue_number",
    "locked": "issue_locked",
    "state": "issue_state",
    "assignees": "assignees_json"
}

In [4]:
# get data with endpoint and headers
def get_data(endpoint, headers):

    # get data using endpoint
    url = "https://api.github.com" + endpoint
    r = requests.get(url, headers=headers)

    # (repo commit endpoint error)
    # if 409 status code then commit history not accessible
    # so do nothing
    if r.status_code == 409:
        return []
    # check for other errors 
    else:
        r.raise_for_status()

        # return json data
        return r.json()

# flatten dictionaries in repo
def flatten_repo_dictionaries(repo):

    # get the current repo dictionary 
    # return empty dict as back up
    owner = repo.get("owner") or {}
    permissions = repo.get("permissions") or {} 
    license = repo.get("license") or {}

    # keep only owner id
    repo["owner_id"] = owner.get("id")

    # flatten repo dicts
    repo["permissions_admin"] = permissions.get("admin")
    repo["permissions_maintain"] = permissions.get("maintain")
    repo["permissions_push"] = permissions.get("push")
    repo["permissions_triage"] = permissions.get("traige")
    repo["permissions_pull"] = permissions.get("pull")

    repo["license_key"] = license.get("key")
    repo["license_name"] = license.get("name")
    repo["license_spdx_id"] = license.get("spdx_id")
    repo["license_url"] = license.get("url")
    repo["license_node_id"] = license.get("node_id")
    
    repo.pop("owner", None)
    repo.pop("permissions", None)
    repo.pop("license", None)

# function to insert ingested data into staging tables
def table_insertion(conn, table_name, all_data):

    # create cursor of DB operations
    cur = conn.cursor()

    # get dict keys for list of columns
    dict_columns = list(all_data[0].keys())

    # query to insert data into DB table
    sql_query = f"""
        INSERT INTO {table_name} ({",".join(dict_columns)})
        VALUES %s
        """

    # convert data into list of tuples for insertion
    data = [
        tuple(row.get(col) for col in dict_columns)
        for row in all_data
    ]

    # load data in DB table
    execute_values(cur, sql_query, data)

    # commit and close cursor
    conn.commit()
    cur.close()

# flatten commit dictionaries while truncating noise
def flatten_repo_commit_dictionaries(commit, repo_id):
        
    # commit dict
    commit_info = commit.get("commit") or {}

    # dicts within commit
    git_author = commit_info.get("author") or {}
    git_committer = commit_info.get("committer") or {}
    tree = commit_info.get("tree") or {}
    verification = commit_info.get("verification") or {}

    # remaining dicts
    author = commit.get("author") or {}
    committer = commit.get("committer") or {}
    parents = commit.get("parents") or []

    # set repo_id
    commit["repo_id"] = repo_id

    # flatten dicts
    commit["message"] = commit_info.get("message")
    commit["comment_count"] = commit_info.get("comment_count")

    commit["git_author_name"] = git_author.get("name")
    commit["git_author_email"] = git_author.get("email")
    commit["git_author_date"] = git_author.get("date")

    commit["git_committer_name"] = git_committer.get("name")
    commit["git_committer_email"] = git_committer.get("email")
    commit["git_committer_date"] = git_committer.get("date")

    commit["author_login"] = author.get("login")
    commit["author_id"] = author.get("id")
    commit["author_type"] = author.get("type")
    commit["author_site_admin"] = author.get("site_admin")
    commit["author_user_view_type"] = author.get("user_view_type")

    commit["committer_login"] = committer.get("login")
    commit["committer_id"] = committer.get("id")
    commit["committer_type"] = committer.get("type")
    commit["committer_site_admin"] = committer.get("site_admin")
    commit["committer_user_view_type"] = committer.get("user_view_type")

    commit["tree_sha"] = tree.get("sha")
    commit["tree_url"] = tree.get("url")

    commit["verified"] = verification.get("verified")
    commit["verification_reason"] = verification.get("reason")
    commit["verification_signature"] = verification.get("signature")
    commit["verification_payload"] = verification.get("payload")
    commit["verified_at"] = verification.get("verified_at")

    # keep only the values in a list
    commit["parent_shas"] = [parent.get("sha") for parent in parents]

    # remove dicts from commit
    commit.pop("commit", None)
    commit.pop("author", None)
    commit.pop("committer", None)
    commit.pop("parents", None)

# flatten the repo issues dicts
def flatten_repo_issues_dictionaries(issue, repo_id):

    # get the dicts in the current issue
    user = issue.get("user") or {}
    sub_issues_summary = issue.get("sub_issues_summary") or {}
    issue_dependencies_summary = issue.get("issue_dependencies_summary") or {}
    reactions = issue.get("reactions") or {}
    assignee = issue.get("assignee") or {}
    closed_by = issue.get("closed_by") or {}

    # to flatten dicts in labels list
    issue_labels = []
    assignees = issue.get("assignees") or []

    # set user id/repo id
    issue["user_id"] = user.get("id")
    issue["repo_id"] = repo_id

    # flatten nested issue dictionaries
    issue['sub_issues_summary_total'] = sub_issues_summary.get('total')
    issue['sub_issues_summary_completed'] = sub_issues_summary.get('completed')
    issue['sub_issues_summary_percent_completed'] = sub_issues_summary.get('percent_completed')

    issue['issue_dependencies_summary_blocked_by'] = issue_dependencies_summary.get('blocked_by')
    issue['issue_dependencies_summary_total_blocked_by'] = issue_dependencies_summary.get('total_blocked_by')
    issue['issue_dependencies_summary_blocking'] = issue_dependencies_summary.get('blocking')
    issue['issue_dependencies_summary_total_blocking'] = issue_dependencies_summary.get('total_blocking')

    issue['reaction_total_count'] = reactions.get('total_count')
    issue['reaction_plus'] = reactions.get('+1')
    issue['reaction_minus'] = reactions.get('-1')
    issue['reaction_laugh'] = reactions.get('laugh')
    issue['reaction_hooray'] = reactions.get('hooray')
    issue['reaction_confused'] = reactions.get('confused')
    issue['reaction_heart'] = reactions.get('heart')
    issue['reaction_rocket'] = reactions.get('rocket')
    issue['reaction_eyes'] = reactions.get('eyes')

    issue['assignee_id'] = assignee.get("id")
    issue['assignee_login'] = assignee.get("login")
    issue['assignee_type'] = assignee.get("type")
    issue['assignee_site_admin'] = assignee.get("site_admin")

    issue["closed_by_id"] = closed_by.get("id")
    issue["closed_by_login"] = closed_by.get("login")
    issue["closed_by_type"] = closed_by.get("type")
    issue["closed_by_site_admin"] = closed_by.get("site_admin")

    # store as structured data (col type JSONB)
    issue["labels_json"] = Json(issue.get("labels") or [])
    issue["assignees_json"] = Json(issue.get("assignees") or [])

    # remove dicts from issue
    issue.pop('user', None)
    issue.pop('sub_issues_summary', None)
    issue.pop('issue_dependencies_summary', None)
    issue.pop('reactions', None)
    issue.pop('pull_request', None)
    issue.pop('pinned_comment', None)
    issue.pop('draft', None)
    issue.pop('type', None)
    issue.pop("assignee", None)
    issue.pop("closed_by", None)
    issue.pop("labels", None)
    issue.pop("assignees", None)

# rename cols to avoid naming issues ( SQL keywords) when creating staging table
def normalize_data_columns(data, rename_map):

    # for each col that needs to be renamed, create the new col and pop the old one
    for old_key, new_key in rename_map.items():
        data[new_key] = data.get(old_key)
        data.pop(old_key, None)

In [5]:
try:
    # pipeline vars
    # get users after id 0
    since = 0
    # number of pages to ingest
    pages = 1

    # pagination size
    users_per_page = 20
    repos_per_page = 10
    commits_per_page = 20
    issues_per_page = 20
    
    # store user/repos/commits/issues data
    all_users_data = []
    all_repos_data = []
    all_commits_data = []
    all_issues_data = []

    # for each page get x users
    for i in range(pages):

        # get users data
        endpoint = f"/users?since={since}&per_page={users_per_page}"

        # returns list
        users = get_data(endpoint, headers)

        # break if users is empty list > stop pagination loop
        if not users:
            break

        # get the usernames to get more data
        logins = [user["login"] for user in users]

        # for each user get more detailed data
        for login in logins:
            
            # endpoint for specific user data using username
            endpoint = f"/users/{login}"

            # returns a dict per user
            user_data = get_data(endpoint, headers)

            # rename user data cols
            normalize_data_columns(user_data, rename_users_map)

            # append user dict to all data
            all_users_data.append(user_data)

            # repo section
            
            # endpoint for specific user repos data using username
            endpoint = f"/users/{login}/repos?per_page={repos_per_page}"

            # returns a list
            repo_data = get_data(endpoint, headers)
            
            # for each repo get the commits/issues for that repo
            for repo in repo_data:

                # flatten repo dictionaries
                flatten_repo_dictionaries(repo)

                # get repo name for url
                repo_name = repo["name"]

                # get repo id for flattening commits/issues dicts
                repo_id = repo['id']

                # rename repo data cols
                normalize_data_columns(repo, rename_repos_map)

                # endpoint for getting commits of a user's specific repo
                endpoint = f"/repos/{login}/{repo_name}/commits?per_page={commits_per_page}"

                # returns a list
                commit_data = get_data(endpoint, headers)

                # flatten the dictionaries in each commit
                for row in commit_data:
            
                    flatten_repo_commit_dictionaries(row, repo_id)

                    # rename commits data cols
                    normalize_data_columns(row, rename_commits_map)

                all_commits_data.extend(commit_data)

                # this section is to ingest issues data
                
                # if repo has issues then call
                if repo["has_issues"] == True:
                    
                    # # endpoint for getting issues of a user's specific repo
                    endpoint = f"/repos/{login}/{repo_name}/issues?per_page={issues_per_page}"

                    # returns list
                    issue_data = get_data(endpoint, headers)

                    # flatten the dictionaries in each issue 
                    for row in issue_data:
                        
                        flatten_repo_issues_dictionaries(row, repo_id)

                        # rename commits data cols
                        normalize_data_columns(row, rename_issues_map)
    
                    # add issue data to all issues data
                    all_issues_data.extend(issue_data)

            # extend repos data list to include repo data
            all_repos_data.extend(repo_data)
        
        # GitHub REST api with users endpoint uses 'since' to get the users where the id is after 'since'
        # therefore, get the last id of the current call to get the next page
        since = users[-1]["id"]

# handle HTTP status errors first, then catch other request failures
except requests.exceptions.HTTPError as e:
    print("HTTP error:", e)
    
except requests.exceptions.RequestException as e:
    print("Other request error:", e)

In [6]:
# data size
print("Number of users:", len(all_users_data))
print("Number of repos:",len(all_repos_data))
print("Number of commits:", len(all_commits_data))
print("Number of issues:", len(all_issues_data))

Number of users: 20
Number of repos: 179
Number of commits: 2271
Number of issues: 236


In [7]:
# connect to postgres DB
conn = psycopg2.connect(f"dbname={DB_NAME} user=postgres password={PASSWORD}")

# insert data into tables
table_insertion(conn, 'users', all_users_data)
table_insertion(conn, 'repos', all_repos_data)
table_insertion(conn, 'commits', all_commits_data)
table_insertion(conn, 'issues', all_issues_data)

# close connection
conn.close()